# climagrid Quickstart

This notebook walks through the three ways to use climagrid:

1. **High-level** — `climagrid.run()` in one line
2. **Mid-level** — individual adapters + joiner
3. **Low-level** — fetch raw data, compute features manually

All three paths produce the same output: a Pandas DataFrame with one row per
(asset, hour), containing environmental observations and engineering stress features.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

import climagrid

print(f"climagrid version: {climagrid.__version__}")

## 1. High-level: `climagrid.run()`

Pass an asset file and a time range. climagrid fetches the data, joins it to
each asset by nearest-neighbor spatial match, and computes all stress features.

In [ ]:
ASSETS = Path("data/sample_assets.csv")   # 45 assets across 9 U.S. states
START  = datetime(2024, 7, 15, tzinfo=timezone.utc)
END    = datetime(2024, 7, 15, 6, tzinfo=timezone.utc)  # 6 hours

df = climagrid.run(
    ASSETS,
    start_dt=START,
    end_dt=END,
    sources=["nasa_power"],   # NASA POWER: global, no API key
    features="all",
)

print(f"Shape: {df.shape}")
print(f"Assets: {df['asset_id'].nunique()}")
print(f"Hours:  {df['timestamp'].nunique()}")
df.head(3)

### What columns are in the output?

In [ ]:
# Index columns: always present
print("Index:", [c for c in df.columns if c in {"asset_id", "timestamp", "lat", "lon"}])

# Environmental observations
print("NASA:  ", [c for c in df.columns if c.startswith("nasa_")])

# Stress features
print("Features:", [c for c in df.columns if c.startswith("feat_")])

### The thermal aging factor (IEEE C57.91)

`feat_thermal_aging_factor` is the Arrhenius FAA: ratio of insulation aging rate
at observed temperature vs. the 110°C reference. Values > 1 mean accelerated aging.

In [ ]:
faa_summary = df.groupby("asset_id")["feat_thermal_aging_factor"].agg(["mean", "max"])
faa_summary.columns = ["mean_FAA", "peak_FAA"]
faa_summary = faa_summary.sort_values("peak_FAA", ascending=False)
print("Top 10 assets by peak thermal aging:")
print(faa_summary.head(10).to_string())

## 2. Mid-level: adapter + joiner

Use individual components when you need more control — e.g., to fetch from
multiple sources and merge them yourself.

In [ ]:
from climagrid.sources.nasa_power import NasaPowerAdapter
from climagrid.sources.base import BoundingBox
from climagrid.assets.registry import AssetRegistry
from climagrid.assets.joiner import AssetEnvironmentJoiner

# 1. Load assets
registry = AssetRegistry(ASSETS)
print(registry)

# 2. Build a bounding box around central Texas
bbox = BoundingBox(min_lat=31.3, max_lat=31.9, min_lon=-97.4, max_lon=-96.9)
print(f"Bbox center: {bbox.center}")

# 3. Fetch raw data
nasa = NasaPowerAdapter()
raw = nasa.fetch(bbox, START, END)
print(f"Raw data shape: {raw.shape}")
raw.head(3)

In [ ]:
# 4. Join to assets by nearest-neighbor spatial match
joiner = AssetEnvironmentJoiner(max_distance_km=200)
joined = joiner.join(registry, raw)
print(f"Joined shape: {joined.shape}")
joined.head(3)

## 3. Low-level: compute a single feature manually

In [ ]:
from climagrid.features.thermal import ThermalStressIndex
from climagrid.features.conductor_sag import ConductorSagIndex

# Add asset_id for grouping (required by ThermalStressIndex)
joined["asset_id"] = joined.get("asset_id", "demo")

# Compute thermal aging — uses nasa_temperature_2m as fallback if hrrr not present
result = ThermalStressIndex().compute(joined)
result = ConductorSagIndex().compute(result)

result[["timestamp", "nasa_temperature_2m",
        "feat_thermal_aging_factor", "feat_conductor_sag_index"]].head(6)

## 4. Export

Wide-form Parquet (one column per feature) is recommended for datasets
larger than 30 days or 100 assets. Long-form is better for ML feature stores.

In [ ]:
from climagrid.outputs import to_parquet, to_csv, to_long_parquet

# Wide-form Parquet
p = to_parquet(df, "/tmp/climagrid_wide.parquet")
print(f"Wide Parquet: {p}  ({p.stat().st_size:,} bytes)")

# Long-form Parquet
p2 = to_long_parquet(df, "/tmp/climagrid_long.parquet")
print(f"Long Parquet: {p2}  ({p2.stat().st_size:,} bytes)")

# Peek at long form
pd.read_parquet(p2).head(6)

## 5. Schema reference

In [ ]:
climagrid.schema_summary()

## 6. Map: asset thermal stress

Plot each asset location colored by its mean thermal aging factor.
Warmer color = more accelerated insulation aging relative to IEEE C57.91 baseline.
This plot is also saved to `docs/assets/quickstart_map.png` for use in the README.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Re-run with TX-only assets so NASA POWER data lands on the right region.
# (The 9-state sample_assets.csv causes a centroid-miss that leaves features NaN.)
df_map = climagrid.run(
    Path('data/tx_assets.csv'),
    start_dt=datetime(2024, 7, 15, tzinfo=timezone.utc),
    end_dt=datetime(2024, 7, 16, tzinfo=timezone.utc),
    sources=['nasa_power'],
    features='all',
)

asset_summary = (
    df_map.groupby('asset_id')
    .agg(lat=('lat', 'first'), lon=('lon', 'first'),
         mean_faa=('feat_thermal_aging_factor', 'mean'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_facecolor('#f0f4f8')
fig.patch.set_facecolor('#ffffff')

faa = asset_summary['mean_faa']
sc = ax.scatter(
    asset_summary['lon'],
    asset_summary['lat'],
    c=faa,
    cmap='plasma',
    vmin=faa.min(),
    vmax=faa.max(),
    s=160,
    alpha=1.0,
    edgecolors='white',
    linewidths=0.8,
    zorder=3,
)

cbar = plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('Mean Thermal Aging Factor (IEEE C57.91)', fontsize=10)

ax.set_xlabel('Longitude', fontsize=10)
ax.set_ylabel('Latitude', fontsize=10)
n = asset_summary.shape[0]
v = climagrid.__version__
ax.set_title(
    f'Grid Asset Thermal Stress \u2014 July 15 2024\n{n} assets \u00b7 Central Texas \u00b7 climagrid {v}',
    fontsize=12, fontweight='bold',
)
ax.grid(True, alpha=0.3, linestyle='--')
ax.tick_params(labelsize=9)
fig.tight_layout()

out = Path('../docs/assets/quickstart_map.png')
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved: {out.resolve()}')
plt.show()